In [1]:
import sys
import os

import tensorflow as tf

os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"
sys.path.append(os.path.abspath(os.path.join(os.path.dirname('variational_ae.py'), '..')))

from tensorflow.keras.datasets import mnist
from variational_ae import VariationalAutoencoder
import numpy as np
from sklearn.model_selection import train_test_split
import h5py

2025-08-09 17:27:31.482122: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-08-09 17:27:31.494520: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1754731651.508787   41224 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1754731651.513047   41224 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1754731651.524695   41224 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [2]:
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"         # Keeps GPU order consistent
os.environ["CUDA_VISIBLE_DEVICES"] = "0"               # Makes only GPU 0 visible (useful even with 1 GPU)

gpus = tf.config.experimental.list_physical_devices("GPU")
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)  # Prevents TF from using all GPU memory at once
    except RuntimeError as e:
        print("Error: ", e)
        exit(-1)

In [3]:
with h5py.File('Dataset/log_spec_data_dataset.h5', 'r') as h5f:
    log_spec_data_train = h5f['train'][:]
    log_spec_data_labels = h5f['label'][:]

In [4]:
log_spec_data_train = log_spec_data_train[..., np.newaxis]

log_spec_x_train, log_spec_x_val = train_test_split(log_spec_data_train, test_size=0.05, random_state=42)

In [5]:
print("Log Spec Train shape:", log_spec_x_train.shape)
print("Log Spec Validation shape:", log_spec_x_val.shape)

Log Spec Train shape: (28500, 256, 64, 1)
Log Spec Validation shape: (1500, 256, 64, 1)


In [6]:
LEARNING_RATE = 0.0005
BATCH_SIZE = 64
EPOCHS = 5

In [7]:
input_shape = log_spec_x_train.shape[1:]
latent_space_dim = 128
decoder_out_filter = 1

In [8]:
# Hyperparameters for the Variational Autoencoder
recon_weight = 1.0  # Weight for the reconstruction loss.
beta = 1.0  # Weight for the KL divergence loss.

In [9]:
autoencoder = VariationalAutoencoder(input_shape, latent_space_dim, decoder_out_filter, recon_weight, beta, conv_layers_config=[
    # {'filters': 64, 'kernel_size': (5, 5), 'strides': (2, 2)},
    # {'filters': 64, 'kernel_size': (3, 3), 'strides': (1, 1)},
    
    # {'filters': 128, 'kernel_size': (3, 3), 'strides': (1, 1)},
    # {'filters': 128, 'kernel_size': (3, 3), 'strides': (1, 1)},
    
    # {'filters': 256, 'kernel_size': (3, 3), 'strides': (1, 1)},
    # {'filters': 256, 'kernel_size': (3, 3), 'strides': (1, 1)},
    
    # {'filters': 384, 'kernel_size': (3, 3), 'strides': (1, 1)},
    
    # {'filters': 512, 'kernel_size': (3, 3), 'strides': (1, 1)},
    
    # {'filters': 256, 'kernel_size': (3, 3), 'strides': (1, 1)},
    
    {'filters': 512, 'kernel_size': (3, 3), 'strides': 2},
    {'filters': 256, 'kernel_size': (3, 3), 'strides': 2},
    {'filters': 128, 'kernel_size': (3, 3), 'strides': 2},
    {'filters': 64, 'kernel_size': (3, 3), 'strides': 2},
    {'filters': 32, 'kernel_size': (3, 3), 'strides': (2, 1)},
])

I0000 00:00:1754731655.825476   41224 gpu_process_state.cc:208] Using CUDA malloc Async allocator for GPU: 0
I0000 00:00:1754731655.826029   41224 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9711 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060, pci bus id: 0000:01:00.0, compute capability: 8.6


In [10]:
autoencoder.compile(learning_rate=LEARNING_RATE)

In [ ]:
autoencoder.summary()

Model: "variational_autoencoder"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_input       │ (None, 256, 64,   │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_conv_layer… │ (None, 128, 32,   │      4,608 │ encoder_input[0]… │
│ (Conv2D)            │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_batch_norm… │ (None, 128, 32,   │      2,048 │ encoder_conv_lay… │
│ (BatchNormalizatio… │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_leaky_relu… │ (None, 128, 32,   │          0 │ encoder_batch_no… │
│ (LeakyReLU)         │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_conv_layer… │ (None, 64, 16,    │  1,179,648 │ encoder_leaky_re… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_batch_norm… │ (None, 64, 16,    │      1,024 │ encoder_conv_lay… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_leaky_relu… │ (None, 64, 16,    │          0 │ encoder_batch_no… │
│ (LeakyReLU)         │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_conv_layer… │ (None, 32, 8,     │    294,912 │ encoder_leaky_re… │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_batch_norm… │ (None, 32, 8,     │        512 │ encoder_conv_lay… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_leaky_relu… │ (None, 32, 8,     │          0 │ encoder_batch_no… │
│ (LeakyReLU)         │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_conv_layer… │ (None, 16, 4, 64) │     73,728 │ encoder_leaky_re… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_batch_norm… │ (None, 16, 4, 64) │        256 │ encoder_conv_lay… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_leaky_relu… │ (None, 16, 4, 64) │          0 │ encoder_batch_no… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_conv_layer… │ (None, 8, 4, 32)  │     18,432 │ encoder_leaky_re… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_batch_norm… │ (None, 8, 4, 32)  │        128 │ encoder_conv_lay… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_leaky_relu… │ (None, 8, 4, 32)  │          0 │ encoder_batch_no… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_flatten_la… │ (None, 1024)      │          0 │ encoder_leaky_re

 Total params: 2,370,304 (9.04 MB)

 Trainable params: 2,367,360 (9.03 MB)

 Non-trainable params: 2,944 (11.50 KB)

: 

In [ ]:
autoencoder.fit(
    x=log_spec_x_train,
    y={"reconstruction": log_spec_x_train}, # Autoencoders typically use the same data for input and output
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(log_spec_x_val, {"reconstruction": log_spec_x_val}), # Validation data for monitoring
    shuffle=True
)

Epoch 1/5


/home/chua/projects/tf217/lib/python3.12/site-packages/keras/src/optimizers/base_optimizer.py:855: UserWarning: Gradients do not exist for variables ['encoder_conv_layer_1/kernel', 'encoder_batch_norm_layer_1/gamma', 'encoder_batch_norm_layer_1/beta', 'encoder_conv_layer_2/kernel', 'encoder_batch_norm_layer_2/gamma', 'encoder_batch_norm_layer_2/beta', 'encoder_conv_layer_3/kernel', 'encoder_batch_norm_layer_3/gamma', 'encoder_batch_norm_layer_3/beta', 'encoder_conv_layer_4/kernel', 'encoder_batch_norm_layer_4/gamma', 'encoder_batch_norm_layer_4/beta', 'encoder_conv_layer_5/kernel', 'encoder_batch_norm_layer_5/gamma', 'encoder_batch_norm_layer_5/beta', 'mu/kernel', 'mu/bias', 'log_variance/kernel', 'log_variance/bias', 'decoder_dense_layer/kernel', 'decoder_dense_layer/bias', 'decoder_conv_transpose_layer_1/kernel', 'decoder_batch_norm_layer_1/gamma', 'decoder_batch_norm_layer_1/beta', 'decoder_conv_transpose_layer_2/kernel', 'decoder_batch_norm_layer_2/gamma', 'decoder_batch_norm_layer